In [ ]:
import pandas as pd

df = pd.read_csv("../data/Tweets.csv")

df = df[["tweet_id", "airline", "airline_sentiment",
         "airline_sentiment_confidence", "negativereason",
         "text", "tweet_created"]]

print(df.shape)
print(df["airline_sentiment"].value_counts())
df.head()

In [ ]:
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def clean_tweet(text):
    text = re.sub(r"http\S+", "", text)   # remove links
    text = re.sub(r"@\w+", "", text)      # remove @mentions
    text = re.sub(r"&amp;", "and", text)  # fix HTML leftovers
    text = re.sub(r"\s+", " ", text)      # tidy extra spaces
    return text.strip()

def vader_label(text):
    score = analyzer.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

df["clean_text"] = df["text"].apply(clean_tweet)
df["vader_raw"] = df["text"].apply(vader_label)
df["vader_clean"] = df["clean_text"].apply(vader_label)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

for col in ["vader_raw", "vader_clean"]:
    print(col, "accuracy:", round(accuracy_score(df["airline_sentiment"], df[col]), 3))

print(classification_report(df["airline_sentiment"], df["vader_clean"]))


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = ["negative", "neutral", "positive"]
cm = confusion_matrix(df["airline_sentiment"], df["vader_clean"], labels=labels)
print(pd.DataFrame(cm, index=[f"true_{l}" for l in labels],
                   columns=[f"pred_{l}" for l in labels]))

# negative tweets that VADER got wrong
missed = df[(df["airline_sentiment"] == "negative") & (df["vader_clean"] != "negative")]
pd.set_option("display.max_colwidth", 200)
missed[["clean_text", "vader_clean"]].sample(10, random_state=1)

In [ ]:
import html, re, time
from transformers import pipeline

def roberta_prep(text):
    text = html.unescape(text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"http\S+", "http", text)
    return text.strip()

MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sentiment_pipe = pipeline("sentiment-analysis", model=MODEL, tokenizer=MODEL)

tests = [
    "@united Now going on 26 minutes on hold.",
    "@SouthwestAir Thanks for the smooth flight, great crew!",
    "@AmericanAir Thanks for losing my bag. Really great service.",
]

results = sentiment_pipe([roberta_prep(t) for t in tests])
for t, r in zip(tests, results):
    print(r["label"], round(r["score"], 2), "|", t)

: 